In [1]:
import numpy as np
import random

In [2]:
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (size-1, size-1)
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        x, y = self.state

        if action == 0:  # up
            x = max(0, x-1)
        elif action == 1:  # down
            x = min(self.size-1, x+1)
        elif action == 2:  # left
            y = max(0, y-1)
        elif action == 3:  # right
            y = min(self.size-1, y+1)

        self.state = (x, y)

        if self.state == self.goal:
            return self.state, 10, True
        else:
            return self.state, -1, False

In [3]:
class QAgent:
    def __init__(self, grid_size, lr=0.1, gamma=0.9, epsilon=0.2):
        self.q_table = np.zeros((grid_size, grid_size, 4))
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon

    def choose_action(self, state):
        if random.uniform(0,1) < self.epsilon:
            return random.randint(0,3)
        x, y = state
        return np.argmax(self.q_table[x, y])

    def update(self, state, action, reward, next_state):
        x, y = state
        nx, ny = next_state

        best_next = np.max(self.q_table[nx, ny])

        self.q_table[x, y, action] += self.lr * (
            reward + self.gamma * best_next - self.q_table[x, y, action]
        )

In [4]:
def train_agent(agent, env, episodes=50):
    for _ in range(episodes):
        state = env.reset()
        done = False

        while not done:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state)
            state = next_state

    return agent.q_table

In [5]:
def federated_average(q_tables):
    return np.mean(q_tables, axis=0)

In [6]:
num_agents = 3
grid_size = 5
rounds = 10

# Initialize global Q-table
global_q = np.zeros((grid_size, grid_size, 4))

agents = [QAgent(grid_size) for _ in range(num_agents)]

for r in range(rounds):
    print(f"Round {r+1}")

    local_q_tables = []

    for i in range(num_agents):
        env = GridWorld(grid_size)

        # Load global model
        agents[i].q_table = global_q.copy()

        # Train locally
        updated_q = train_agent(agents[i], env, episodes=50)
        local_q_tables.append(updated_q)

    # Server aggregates
    global_q = federated_average(local_q_tables)

print("Federated Reinforcement Learning Complete!")

Round 1
Round 2
Round 3
Round 4
Round 5
Round 6
Round 7
Round 8
Round 9
Round 10
Federated Reinforcement Learning Complete!


In [7]:
def test_agent(global_q, grid_size=5, max_steps=50):
    env = GridWorld(grid_size)
    state = env.reset()

    path = [state]
    total_reward = 0

    for step in range(max_steps):
        x, y = state
        action = np.argmax(global_q[x, y])  # greedy action

        next_state, reward, done = env.step(action)
        path.append(next_state)
        total_reward += reward

        state = next_state

        if done:
            break

    return path, total_reward


# Run test
path, reward = test_agent(global_q)

print("🧭 Path taken by agent:")
print(path)

print("\n🏁 Total Reward:", reward)

if path[-1] == (4,4):
    print("\n✅ Goal reached successfully!")
else:
    print("\n❌ Goal not reached")

🧭 Path taken by agent:
[(0, 0), (0, 1), (1, 1), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]

🏁 Total Reward: 3

✅ Goal reached successfully!
